# Arquitetura Bronze
Este notebook implementa a **camada Bronze** da arquitetura medalhão. 

## Fontes de dados
| Fonte | Origem | Tabela de destino |
|---|---|---|
| Informações dos filmes | CSV (Volume) | `bronze.tb_movies_info` |
| Dados financeiros | CSV (Volume) | `bronze.tb_movies_financials` |
| Métricas | CSV (Volume) | `bronze.tb_movies_metrics` |
| Créditos e tags | CSV (Volume) | `bronze.tb_credits_and_tags` |
| Reviews | CSV (Volume) | `bronze.tb_movies_reviews` |
| Cotação do dólar (PTAX) | API do Banco Central (Olinda) | `bronze.tb_cotacao_dolar` |

## Estrutura do notebook
1. Configuração do catálogo e da lista de arquivos
2. Inspeção rápida dos CSVs
3. Ingestão dos CSVs em tabelas Delta
4. Teste de conectividade de rede
5. Ingestão da cotação do dólar via API (com fallback para mock)

## Princípios adotados
- **Dados crus:** todas as colunas são lidas como `string`, sem conversão de tipos. A tipagem fica para a Silver.
- **Auditoria:** toda tabela recebe a coluna `ingestion_datetime`, com o momento da carga.
- **Formato Delta:** garante transações ACID, histórico de versões (*time travel*) e leitura eficiente pelas camadas seguintes.

## 1. Configuração inicial

Este bloco prepara o ambiente para a ingestão:

- **`functions as F`:** importa as funções do PySpark (aqui, usada para `current_timestamp()`).
- **`CATALOGO`:** nome do catálogo do Unity Catalog onde ficam as tabelas. O comando `USE CATALOG` o define como padrão da sessão, então `bronze.tb_movies_info` é resolvido como `atividade_2_cinedata.bronze.tb_movies_info`.
- **`caminho`:** pasta do **Volume** (`/Volumes/<catálogo>/raw/inputs/`) onde os CSVs brutos estão armazenados.
- **`arquivos`:** lista de tuplas `(arquivo de origem, tabela de destino)`. Ela guia a ingestão: para incluir uma nova fonte, basta adicionar uma linha aqui, sem alterar a lógica.

In [0]:
from pyspark.sql import functions as F

CATALOGO = "atividade_2_cinedata"
spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze") 

arquivos=[ 
    ("movies_info_TMDB_IMDB.csv", "bronze.tb_movies_info"),
    ("movies_financials_IMDB_TMDB.csv", "bronze.tb_movies_financials"),
    ("movies_metrics_IMDB_TMDB.csv", "bronze.tb_movies_metrics"),
    ("credits_and_tags_IMDB_TMDB.csv", "bronze.tb_credits_and_tags"),
    ("movies_reviews.csv", "bronze.tb_movies_reviews"),
    ]

## 2. Ingestão dos CSVs para tabelas Delta

A função `ingerir_bronze(arquivo, tabela)` é o núcleo da camada Bronze. Para cada arquivo, ela executa três etapas:

1. **Leitura:** lê o CSV do Volume com as mesmas opções da inspeção (`header`, `multiline`, `escape`), sem alterar o conteúdo.

1. **Auditoria:** adiciona a coluna `ingestion_datetime` com `F.current_timestamp()`, registrando quando cada carga ocorreu. Isso permite rastrear a origem temporal dos dados e identificar cargas duplicadas.

3. **Escrita:** grava o resultado como tabela **Delta** gerenciada (`saveAsTable`) no modo `append`.

Ao final, imprime o nome da tabela e a quantidade de linhas gravadas, o que serve de verificação rápida da carga.

O `for` percorre a lista `arquivos` e chama a função para cada par `(arquivo, tabela)`, ingerindo as cinco fontes de CSV em sequência.

In [0]:
def ingerir_bronze(arquivo, tabela):
    df_raw = (spark.read
              .option("header", True)
              .option("multiline", True)
              .option("escape", '"')
              .csv(f"{caminho}{arquivo}"))
    
    # Auditoria
    df_bronze = df_raw.withColumn("ingestion_datetime", F.current_timestamp())

    # Escrita
    (df_bronze.write
        .format("delta")
        .mode("append")
        .saveAsTable(tabela))
    
    print(f"{tabela}: {df_bronze.count()} linhas")

for arq, tab in arquivos:
    ingerir_bronze(arq, tab)

## 4. Ingestão da cotação do dólar

Este bloco consome a API **Olinda/PTAX** do Banco Central, que retorna a cotação diária do dólar, e grava o resultado em `bronze.tb_cotacao_dolar`.

### 4.1 Parâmetros via widgets
Os widgets `data_inicio` e `data_fim` permitem escolher o período da consulta sem editar o código.
- **Formato:** `MM-DD-AAAA` (padrão exigido pela API).
- **Padrão:** os últimos 30 dias até hoje.
- A API **não retorna cotações de fins de semana e feriados**. Essas lacunas serão tratadas na Silver (*forward fill*).

### 4.2 Montagem da requisição
A URL é construída manualmente porque o Olinda exige os caracteres `@` e `$` de forma literal. O dicionário `params` reúne os parâmetros da consulta:

| Parâmetro | Função |
|---|---|
| `@dataInicial` / `@dataFinalCotacao` | Período da consulta |
| `$format` | Formato da resposta (`json`) |
| `$orderby` | Ordenação por data/hora da cotação, crescente |
| `$top` / `$skip` | Paginação (até 1000 registros por chamada) |

### 4.3 Estratégia de fallback (API real → mock)
O código tenta primeiro a **API real**. Se a chamada falhar (rede bloqueada, erro 500 do servidor) ou retornar uma lista vazia, aciona `gerar_mock()`.

A função `gerar_mock()` simula a resposta da API com a **mesma estrutura original**:
- Gera cotações apenas em **dias úteis**, como a API real.
- Parte de um valor inicial e aplica uma pequena variação aleatória (`random.seed(42)` garante reprodutibilidade).
- Os valores são **fictícios** e servem apenas para manter o pipeline funcional.

A variável `fonte` registra a origem dos dados (`API BCB` ou `MOCK`) e é impressa junto com a quantidade de cotações, para deixar clara a procedência da carga.

### 4.4 Gravação
Os registros viram um DataFrame com schema explícito (`dataHoraCotacao` e `cotacaoCompra`, ambos `string`, mantendo o dado cru), recebem a coluna `ingestion_datetime` e são gravados em Delta no modo `append`. O `display()` final mostra o resultado.

In [0]:
import requests
import random
from datetime import datetime, timedelta, date
from urllib.parse import quote

# Widgets: período da consulta, formato MM-DD-AAAA (padrão da API).
# Padrão: 30 dias corridos até hoje (a API não retorna fins de semana e feriados).
hoje = date.today()
dbutils.widgets.text("data_inicio", (hoje - timedelta(days=30)).strftime("%m-%d-%Y"), "Data início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", hoje.strftime("%m-%d-%Y"), "Data fim (MM-DD-AAAA)") 

data_inicio_formatada = dbutils.widgets.get("data_inicio")
data_fim_formatada = dbutils.widgets.get("data_fim")
data_inicial = datetime.strptime(data_inicio_formatada, "%m-%d-%Y").date()
data_final = datetime.strptime(data_fim_formatada, "%m-%d-%Y").date()

# URL no formato do enunciado. Montada à mão porque o Olinda exige '@' e '$' literais
url_base = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
)

params = {
    "@dataInicial": f"'{data_inicial:%m-%d-%Y}'",
    "@dataFinalCotacao": f"'{data_final:%m-%d-%Y}'",
    "$format": "json",
    "$orderby": "dataHoraCotacao asc",
    "$top": 1000,
    "$skip": 0
}

# Query string manual: mantém '@', '$' e aspas simples literais (o requests os
# converteria para %40, %24 e %27 se usássemos params=params).
SAFE_CHARS = "@$'"
query_string = "&".join(
    f"{k}={quote(str(v), safe=SAFE_CHARS)}" for k, v in params.items()
)
url = f"{url_base}?{query_string}"

def gerar_mock(inicio, fim, cotacao_inicial=5.40, semente=42):
    """MOCK: simula a resposta da API PTAX com a MESMA estrutura do JSON original.
    Os valores são FICTÍCIOS e não representam a cotação oficial do BCB.
    Dias úteis recebem uma cotação com pequena variação; sábados e domingos ficam de
    fora, como na API real, para que o forward fill do Silver tenha lacunas a preencher."""
    random.seed(semente)
    cotacao, registros, dia = cotacao_inicial, [], inicio
    while dia <= fim:
        if dia.weekday() < 5:          # 0-4 = segunda a sexta
            cotacao = round(cotacao + random.uniform(-0.05, 0.05), 4)
            registros.append({
                "dataHoraCotacao": f"{dia:%Y-%m-%d} 13:05:00.000",   # formato do Olinda
                "cotacaoCompra": cotacao,
            })
        dia += timedelta(days=1)
    return registros

# API real primeiro; mockup só se a rede estiver bloqueada ou a API falhar.
try:
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    registros = resp.json()["value"]
    if not registros:
        raise ValueError("API retornou lista vazia para o período")
    fonte = "API BCB (dados reais)"
except (requests.exceptions.RequestException, ValueError) as e:
    # Free Edition bloqueia olinda.bcb.gov.br (DNS não resolve); o servidor também
    # pode responder 500. Em ambos os casos usamos o mockup.
    print(f"Falha na API: {type(e).__name__}")
    registros = gerar_mock(data_inicial, data_final)
    fonte = "MOCK (valores fictícios)"

print(f"FONTE DOS DADOS: {fonte} | {len(registros)} cotações")

df_cotacao = (
    spark.createDataFrame(
        [(r["dataHoraCotacao"], str(r["cotacaoCompra"])) for r in registros],
        schema="dataHoraCotacao string, cotacaoCompra string",
    )
    .withColumn("ingestion_datetime", F.current_timestamp())
)

df_cotacao.write.format("delta").mode("append").saveAsTable("bronze.tb_cotacao_dolar")
display(df_cotacao)